In [1]:
from uuid import UUID
from db.repositories.videos import VideoRepository, VideoDataLoader
from db.repositories.topics import TopicRepository, TopicLoader
from db.conf import create_db_engine, get_async_session
from core.agents.series import VideoSeriesAnalyzer
from core.agents.tpl import TemplateManager

engine = create_db_engine()
db = get_async_session(engine)

tpm_mgr = TemplateManager()
agent = VideoSeriesAnalyzer(tpm_mgr)


In [2]:
author_id = UUID("8f37db5e-a7f6-11f0-8100-6fca046f2af4")

async with db() as session:
  video_repo = VideoRepository(session)
  video_loader = VideoDataLoader(author_id, video_repo)
  
  topic_repo = TopicRepository(session)
  topic_loader = TopicLoader(topic_repo)
  
  video_data = await video_loader.load_video_data(max_videos=10)
  topic_data = await topic_loader.load_topics()
  
  result = await agent.analyze(video_data, topic_data)
  
result

2025-10-15 14:27:21,398 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-10-15 14:27:21,399 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-15 14:27:21,403 INFO sqlalchemy.engine.Engine select current_schema()
2025-10-15 14:27:21,403 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-15 14:27:21,405 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-10-15 14:27:21,405 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-15 14:27:21,407 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-15 14:27:21,414 INFO sqlalchemy.engine.Engine SELECT videos.id, videos.url, videos.source, videos.author_id, videos.uploaded_at, videos.likes, videos.views, videos.comments, videos.revision, videos.extra_data, videos.created_at, videos.updated_at, videos.processed_at, videos.processing_error 
FROM videos 
WHERE videos.author_id = $1::UUID AND (videos.processing_error IS false OR videos.processing_error IS NULL) ORDER BY videos.uploaded_at DESC 
 LIMIT $2::INTEGER
2025-

TopicsResponse(topics=[TopicDecision(canonical_topic='Romance and Couples Content', decision='existing', topic_id=UUID('12b0c58c-a8e1-11f0-bdbb-172637d32819'), proposed_topic_name=None, supporting_videos=[SupportingVideo(video_id=UUID('9ab7cade-a99a-11f0-9cd1-2302fd6cbd9e'), evidence='montage showcasing various romantic moments between a man and a woman'), SupportingVideo(video_id=UUID('9ab27106-a99a-11f0-8501-8bbcdbcfd689'), evidence='couple silhouetted against a city skyline'), SupportingVideo(video_id=UUID('524253ec-a9ad-11f0-90a9-a795ebd97acd'), evidence='montage showcasing various romantic or intimate moments between a man and a woman')], alternates_considered=[], confidence=0.95), TopicDecision(canonical_topic='Pet Ownership and Dog Behavior', decision='new', topic_id=None, proposed_topic_name='Pet Ownership and Dog Behavior', supporting_videos=[SupportingVideo(video_id=UUID('9ab67760-a99a-11f0-8501-4f96789cf4d3'), evidence='woman and a small, curly-haired brown dog, likely a poo